In [ ]:
train_ds = tf.data.Dataset.from_tensor_slices((X_train.to_numpy(),y_train.to_numpy()))

test_ds = tf.data.Dataset.from_tensor_slices((X_test.to_numpy(),y_test.to_numpy()))

type(train_ds), type(test_ds)

In [ ]:
train_ds = train_ds.cache()
train_ds = train_ds.prefetch(tf.data.AUTOTUNE)

test_ds = test_ds.cache()
test_ds = test_ds.prefetch(tf.data.AUTOTUNE)

In [ ]:
train_ds = train_ds.shuffle(buffer_size=X_train.shape[0]).batch(BATCH_SIZE)

test_ds = test_ds.shuffle(buffer_size=X_test.shape[0]).batch(BATCH_SIZE)


In [ ]:
### Normalization of Data in TensorFlow


normalizer = tf.keras.layers.Normalization(axis=-1)
normalizer.adapt(X_train.to_numpy())
normalizer.mean.numpy()

In [ ]:
inputs = tf.keras.Input(shape=(X_train.shape[1],))

x = normalizer(inputs)

x = tf.keras.layers.Dense(16,activation='relu')(x) #tf.nn.relu
x = tf.keras.layers.Dense(8,activation='relu')(x)
outputs = tf.keras.layers.Dense(4)(x)
model = tf.keras.Model(inputs=inputs,outputs=outputs)

In [ ]:
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
optimizer = tf.keras.optimizers.Adam(learning_rate=ALPHA)

model.compile(optimizer=optimizer,loss=loss_fn,metrics=['accuracy'])

In [ ]:
history = model.fit(train_ds,
                    epochs=EPOCHS,
                    batch_size=BATCH_SIZE,
                    validation_data=test_ds)

In [ ]:
# Feature Importance

for layer in model.layers:
    if isinstance(layer,tf.keras.layers.Dense):
        first_kernel = layer
        break

weights ,bias = first_kernel.get_weights()
weights.shape, bias.shape

feature_importance = np.abs(weights).sum(axis = 1)
feature_df = pd.DataFrame({'feature':X_train.columns,
                            'importance':feature_importance})
feature_df = feature_df.sort_values(by='importance', ascending = False)
fig, ax = plt.subplots()

ax.barh(feature_df['feature'],feature_df['importance'])
plt.gca().invert_yaxis()




In [ ]:
y_true ,y_pred = [],[]
for features,labels in train_ds:
    pred = model(features,training=False)
    pred = pred.numpy().argmax(axis=1)
    y_pred.append(pred)
    y_true.append(labels.numpy())

y_true = np.concatenate(y_true)
y_pred = np.concatenate(y_pred)
accuracy_score(y_true,y_pred)
# y_true.shape,y_pred.shape


In [ ]:
cm = confusion_matrix(y_train,y_pred.argmax(axis = 1))
disp = ConfusionMatrixDisplay(cm,display_labels=[1,2,3])
fig,ax = plt.subplots(figsize = (4,4))

disp.plot(ax = ax,cmap = 'Blues',xticks_rotation='vertical',colorbar=False)

In [ ]:
class FifaDataset(Dataset):

    def __init__(self,X,y):
        super().__init__()
        self.X = torch.tensor(X,dtype = torch.float32)
        self.y = torch.tensor(y,dtype = torch.long)

    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, index):
        return self.X[index],self.y[index]


In [ ]:
from torch.utils.data import DataLoader

train_dataset = FifaDataset(X_train,y_train)
train_loader = DataLoader(dataset = train_dataset,batch_size = BATCH_SIZE, shuffle = True)

for batch_idx,(data,target) in enumerate(train_loader):
    print(f'Batch: {batch_idx + 1}:', end = '')
    print(f'Data: {data.shape}:', end = '')
    print(f'Target: {target.shape}:')

test_dataset = FifaDataset(X_test,y_test.to_numpy()) # <---- we did .tonumpy() idk why but we did 
test_loader = DataLoader(dataset = test_dataset,batch_size = BATCH_SIZE, shuffle = True)

for batch_idx,(data,target) in enumerate(test_loader):
    print(f'Batch: {batch_idx + 1}:', end = '')
    print(f'Data: {data.shape}:', end = '')
    print(f'Target: {target.shape}:')



In [ ]:
class FifaModel(nn.Module):
    ''' Layers:
        33 --> 16 --> 8 --> 4  --] 3-layers
        33 --> 8 --> 4         --] 2-layers
    '''
    def __init__(self, input_dim):
        super(FifaModel,self).__init__()

        self.layer1 = nn.Linear(input_dim,16)
        self.activ1 = nn.ReLU()

        self.layer2 = nn.Linear(16,8)
        self.activ2 = nn.ReLU()

        self.layer3 = nn.Linear(8,4)


    def forward(self,x):
        x = self.layer1(x)
        x = self.activ1(x)

        x = self.layer2(x)
        x = self.activ2(x)

        x = self.layer3(x)
        return x        
    

model = FifaModel(input_dim=X_train.shape[1]).to(device=device)# i.e, 33


In [ ]:
from unittest import TestLoader


loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(),lr = ALPHA)

# these are the things we usually track
loss,tloss, n_epoch, acc ,tacc = [],[],[],[],[]

for epoch in range(EPOCHS):
    model.train()

    epoch_loss  = 0
    epoch_acc   = 0
    tepoch_loss = 0
    tepoch_acc  = 0
    
    # Training
    for batch_idx, (train_X,train_y) in enumerate (train_loader):
        train_X,train_y = train_X.to(device),train_y.to(device)
        predict_prob    = model(train_X)
        batch_loss      = loss_fn(predict_prob,train_y)
        epoch_loss     += (batch_loss - epoch_loss) / (batch_idx + 1)

        optimizer.zero_grad() # because wedont want to crry it to next batch.
        batch_loss.backward()
        optimizer.step()

        _,y_pred = torch.max(predict_prob,1) # predict Class
        
        batch_acc = accuracy_score(train_y.cpu().numpy(),y_pred.data.cpu())
        epoch_acc += (batch_acc - epoch_acc) / (batch_idx + 1)
    
    loss.append(epoch_loss.data.item())
    acc.append(epoch_acc)

    model.eval() # stop learning

    # Testing
    with torch.inference_mode():

        for batch_idx, (test_X,test_y) in enumerate (test_loader):
            test_X,test_y = test_X.to(device),test_y.to(device)
            predict_prob_tst    = model(test_X)
            tbatch_loss      = loss_fn(predict_prob_tst,test_y)
            tepoch_loss     += (tbatch_loss - tepoch_loss) / (batch_idx + 1)

            _,y_pred = torch.max(predict_prob_tst,1) # predict Class
            
            tbatch_acc = accuracy_score(test_y.cpu().numpy(),y_pred.data.cpu())
            tepoch_acc += (tbatch_acc - tepoch_acc) / (batch_idx + 1)
        
        tloss.append(tepoch_loss.data.item())
        tacc.append(tepoch_acc)

    n_epoch.append(epoch)

    if epoch %20 == 0:
        fmtStr = 'Epoch : {:5d}/{:5d} | Loss: {:.5f}/{:.5f} | Acc : {:.5f}/{:.5f}'
        print(fmtStr.format(epoch,EPOCHS,
                            epoch_loss.data.item(),tepoch_loss.data.item(),
                            epoch_acc,tepoch_acc))




y_true,y_pred = [],[]

model.eval()

with torch.inference_mode():
    for batchidx,(train_X,train_y) in enumerate(train_loader):
        train_X = train_X.to(device) # no need for train_y
        pred = model(train_X)
        y_pred.extend(torch.argmax(pred ,dim = 1).cpu().numpy())
        y_true.extend(train_y)


print(f'Train Acc: {accuracy_score(y_true,y_pred)}')





In [ ]:
# wine dataset

model = tf.keras.Sequential([
    tf.keras.Input(shape = (X_train.shape[1],)),
    tf.keras.layers.Dense(8,activation='relu'),
    tf.keras.layers.Dense(len(le.classes_)), # we will not use any classficiaton in last layer
])

In [ ]:
#fashion mnist
X_train = X_train / 255.0

X_test = X_test / 255.0

train_ds = tf.data.Dataset.from_tensor_slices((X_train,y_train))
test_ds = tf.data.Dataset.from_tensor_slices((X_test,y_test))


train_ds = train_ds.cache()
train_ds = train_ds.prefetch(tf.data.AUTOTUNE)

test_ds = test_ds.cache()
test_ds = test_ds.prefetch(tf.data.AUTOTUNE)

train_ds = train_ds.cache()
train_ds = train_ds.prefetch(tf.data.AUTOTUNE)

test_ds = test_ds.cache()
test_ds = test_ds.prefetch(tf.data.AUTOTUNE)

In [ ]:
optimzer = tf.keras.optimizers.Adam(learning_rate = ALPHA)
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
regularizer = tf.keras.regularizers.L2(0.05)

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape = (X_train.shape[1],)),

    
    tf.keras.layers.Dense(392),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Activation(activation = 'relu'),
    
    tf.keras.layers.Dense(192),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Activation(activation = 'relu'),

    tf.keras.layers.Dense(98),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Activation(activation = 'relu'),

    tf.keras.layers.Dense(49),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Activation(activation = 'relu'),

    tf.keras.layers.Dense(24),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Activation(activation = 'relu'),

    tf.keras.layers.Dense(10,)
])